# CNN Model Training - Crop Disease Detection
## Building and Training Convolutional Neural Network

In [ ]:
import pandas as pd
import numpy as np
import cv2
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/processed/dataset_metadata.csv')
print(f"Total images: {len(df):,}")
print(f"Classes: {df['class_label'].nunique()}")

## 2. Define CNN Architecture

In [ ]:
class CropDiseaseCNN(nn.Module):
    def __init__(self, num_classes):
        super(CropDiseaseCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

print("CNN Architecture defined")

## 3. Create Dataset Class

In [ ]:
class CropDataset(Dataset):
    def __init__(self, image_paths, labels):
        self.image_paths = image_paths
        self.labels = labels
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (128, 128))
        img = img.transpose(2, 0, 1)
        img = img / 255.0
        return torch.FloatTensor(img), self.labels[idx]

print("Dataset class created")

## 4. Prepare Data

In [ ]:
y_labels = df['class_label'].values
y, labels = pd.factorize(y_labels)
num_classes = len(labels)

print(f"Number of classes: {num_classes}")

X_train, X_test, y_train, y_test = train_test_split(
    df['image_path'].values, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

## 5. Create Data Loaders

In [ ]:
train_dataset = CropDataset(X_train, y_train)
test_dataset = CropDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

## 6. Initialize Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = CropDiseaseCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

## 7. Training Loop

In [ ]:
epochs = 20
train_losses = []
train_accs = []
test_accs = []
best_acc = 0.0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for images, labels_batch in pbar:
        images, labels_batch = images.to(device), labels_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels_batch.size(0)
        correct += (predicted == labels_batch).sum().item()
        
        pbar.set_postfix({'loss': f'{running_loss/total:.4f}', 'acc': f'{100*correct/total:.2f}%'})
    
    train_acc = 100 * correct / total
    train_losses.append(running_loss / len(train_loader))
    train_accs.append(train_acc)
    
    # Evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels_batch in test_loader:
            images, labels_batch = images.to(device), labels_batch.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels_batch.size(0)
            correct += (predicted == labels_batch).sum().item()
    
    test_acc = 100 * correct / total
    test_accs.append(test_acc)
    
    print(f"Epoch {epoch+1}: Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")
    
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), '../models/crop_disease_cnn.pth')
        print(f"✓ Model saved (Best Acc: {best_acc:.2f}%)")

print(f"\nTraining complete! Best accuracy: {best_acc:.2f}%")

## 8. Plot Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Training Loss', color='blue')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss Over Time')
ax1.legend()
ax1.grid(True)

ax2.plot(train_accs, label='Training Accuracy', color='green')
ax2.plot(test_accs, label='Test Accuracy', color='orange')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy Over Time')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 9. Save Label Encoder

In [ ]:
with open('../models/label_encoder.pkl', 'wb') as f:
    pickle.dump(labels, f)

print("Label encoder saved!")

## 10. Test Predictions

In [ ]:
model.eval()
sample_images = X_test[:6]
sample_labels = y_test[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (img_path, true_label) in enumerate(zip(sample_images, sample_labels)):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    img_tensor = cv2.resize(img_rgb, (128, 128))
    img_tensor = img_tensor.transpose(2, 0, 1) / 255.0
    img_tensor = torch.FloatTensor(img_tensor).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1)[0]
        pred_label = torch.argmax(probs).item()
        confidence = probs[pred_label].item()
    
    axes[idx].imshow(img_rgb)
    axes[idx].set_title(f"True: {labels[true_label][:20]}...\nPred: {labels[pred_label][:20]}...\nConf: {confidence:.2%}", 
                       fontsize=8)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()